In [1]:
%load_ext autoreload
%autoreload 2

#### I compared the station_name but didn't compare the native_id. While the raws insertaion based on native_id to recongnize station. 

Because I didn't compare native_id before inserting, and the native_id exist many mistach, thus caused some confusion.

It can be mainly devided into two parts, the first batch is different id, same station_name; the second batch is different id, different 


Thus, only the data with same native_id has been inserted and merged; data with different native_id are recognized as different stations.

### Match stations, using station name, lat, lon. The name matching in Raw data meta_data form, in the folder and in the database

In [2]:
from crmprtd import Row
import logging
import os
import pickle
import pandas as pd
import numpy as np
from natsort import natsorted
from natsort import natsort_keygen
from pprint import pprint
from collections import Counter
from collections import defaultdict

import re
from rapidfuzz import fuzz
from crmprtd import infer
from fern_func import *
# from sql_func import *
import sqlalchemy as sa
from fern_raw_db_dompare import *

In [3]:
engine = sa.create_engine("postgresql://tongli1997@db.pcic.uvic.ca:5433/crmp?keepalives=1&keepalives_idle=300&keepalives_interval=300&keepalives_count=9&passfile=/workspaces/crmprtd/.pgpass", echo=False)
session = Session(engine)

In [4]:
# --- Main workflow ---
# --- read data ---
meta_path = '/workspaces/crmprtd/fern/FERNNorth2025_insert/tables/20241125 MeteorologicalNetworks-FERN-VF-shared.xlsx'

df_station = pd.read_excel(meta_path)
df_station = df_station.sort_values(by='station_name', key=natsort_keygen())


# --- output folder ---
output_folder = '/workspaces/crmprtd/fern/FERNNorth2025_insert/output/'
os.makedirs(output_folder, exist_ok=True)


In [5]:

native_ids = df_station['native_id'].tolist()
station_names = df_station['station_name'].tolist()

query = sa.text("""
    SELECT DISTINCT m.station_name, s.native_id
    FROM meta_history AS m
    JOIN meta_station AS s ON m.station_id = s.station_id
    WHERE s.network_id = 11
      AND m.station_name = ANY(:station_names)
""")

with engine.connect() as conn:
    existing_stations = pd.read_sql(query, conn, params={"station_names": station_names})

print("Stations found in DB:")
existing_stations

Stations found in DB:


,station_name,native_id
0,Atlin School,AtlSch
1,BarrenWx,Barren
2,BlackhawkWx,Blackhawk
3,BoulderWx,Boulder Creek
4,BowronPit,BowronPit
5,BulkleyWx,PGTIS1
6,Canoe Mountain Stn,Canoe Mountain Stn
7,Caribou Pass,CaribouPass
8,ChapmanWx,Chapman
9,ChiefLakeWx,ChiefWx


In [6]:
import os
import pandas as pd
import re
data_path = '/workspaces/crmprtd/fern/FERNNorth2025_VF_data'

# --- Load the CSV ---
compare_file_path = os.path.join(output_folder, "Fern_raw_db_station_matched.csv")
compare_summary = pd.read_csv(compare_file_path)

# --- Step 1: Define base variable (remove trailing digits) ---
def base_variable(var):
    return re.sub(r'\s*\d+$', '', var).strip()

compare_summary['base_variable'] = compare_summary['variable'].apply(base_variable)

# --- Step 2: Identify duplicates per station + base_variable + unit_raw ---
group_keys = ['station_name', 'base_variable', 'unit_raw']

# Rows that are duplicates (appear more than once in the group)
dup_items = (
    compare_summary
    .groupby(group_keys)
    .filter(lambda x: len(x) > 1)
    .reset_index(drop=True)
)

# --- Step 3: Identify unique rows (not in duplicates) ---
# Safer approach using merge instead of relying on index
dup_keys = dup_items[group_keys].drop_duplicates()
unique_items = (
    compare_summary
    .merge(dup_keys, on=group_keys, how='left', indicator=True)
    .query('_merge == "left_only"')  # only rows not in dup_items
    .drop(columns='_merge')
)

# --- Step 4: Sort unique items for readability ---
unique_items = unique_items.sort_values(by=['station_name', 'variable']).reset_index(drop=True)


### Loop for all stations and variables

In [7]:
import os
import pandas as pd

def compare_csv_vs_db(row, data_path, engine):
    """
    Compare one CSV variable with the database.
    Returns a dataframe with Date, csv_value, db_value, diff, match.
    """
    csv_file_name = row['station_file_name'] + '.csv'
    csv_path = os.path.join(data_path, csv_file_name)

    # Load CSV
    df_data = safe_read_csv(csv_path)

    # Time column
    time_col = 'Date' if 'Date' in df_data.columns else df_data.columns[0]
    df_data[time_col] = pd.to_datetime(df_data[time_col], errors='coerce')

    # Variable column
    variable_name = row['variable'] + ', ' + row['unit_origin']
    df_subset = df_data[[time_col, variable_name]].copy()
    df_subset.rename(columns={time_col: 'Date', variable_name: 'csv_value'}, inplace=True)

    # Pull DB data
    db_var_name = row['db_var_match']
    db_native_id = row['native_id_raw']
    query = """
    SELECT o.obs_time, o.datum
    FROM obs_raw o
    JOIN meta_history h ON o.history_id = h.history_id
    JOIN meta_station s ON h.station_id = s.station_id
    JOIN meta_vars v ON o.vars_id = v.vars_id
    WHERE s.native_id = %s
      AND v.net_var_name = %s
      AND o.obs_time BETWEEN %s AND %s
    ORDER BY o.obs_time
    """
    params = (
        db_native_id,
        db_var_name,
        df_subset['Date'].min(),
        df_subset['Date'].max()
    )
    df_db = pd.read_sql(query, engine, params=params)
    df_db.rename(columns={'obs_time': 'Date', 'datum': 'db_value'}, inplace=True)

    # Merge CSV & DB
    df_compare = pd.merge(df_subset, df_db, on='Date', how='outer')

    # Keep rows where at least one value exists
    df_compare = df_compare[~(df_compare['csv_value'].isna() & df_compare['db_value'].isna())].copy()

    # Differences & match
    # --- Ensure numeric types for comparison ---
    df_compare['csv_value'] = pd.to_numeric(df_compare['csv_value'], errors='coerce')
    df_compare['db_value']  = pd.to_numeric(df_compare['db_value'], errors='coerce')

    # Differences & match
    df_compare['diff'] = df_compare['csv_value'] - df_compare['db_value']

    # Consider them matching if both are NaN or difference is very small
    df_compare['match'] = df_compare['diff'].fillna(0).abs() < 0.01  # tolerance for floats

    return df_compare



In [8]:
from sqlalchemy import text
import pandas as pd

network_id = 11

for i, row in unique_items.iloc[0:].iterrows():

    print(f"\n🔍 Comparing {row['station_name']} — {row['variable']}")

    df_comp = compare_csv_vs_db(row, data_path, engine)
    df_update = df_comp[~df_comp['match']].copy()

    if df_update.empty:
        print("✅ No updates needed.")
        continue

    # Only keep what we need
    temp_df = pd.DataFrame({
        "obs_time": df_update["Date"],
        "new_value": df_update["csv_value"]
    })

    with engine.begin() as conn:

        # 1️⃣ Create temp table
        conn.execute(text("""
            CREATE TEMP TABLE temp_updates (
                obs_time timestamp,
                new_value double precision
            ) ON COMMIT DROP
        """))

        # 2️⃣ Bulk insert using COPY (FAST)
        temp_df.to_sql(
            "temp_updates",
            conn,
            if_exists="append",
            index=False,
            method="multi"
        )

        # 3️⃣ Single UPDATE JOIN
        result = conn.execute(text("""
            UPDATE obs_raw o
            SET datum = t.new_value
            FROM temp_updates t,
                 meta_history h,
                 meta_station s,
                 meta_vars v
            WHERE o.obs_time = t.obs_time
              AND o.history_id = h.history_id
              AND h.station_id = s.station_id
              AND o.vars_id = v.vars_id
              AND s.network_id = :network_id
              AND s.native_id = :native_id
              AND v.net_var_name = :var_name
              AND o.datum IS DISTINCT FROM t.new_value
        """), {
            "network_id": network_id,
            "native_id": row["native_id_raw"],
            "var_name": row["db_var_match"]
        })

    print(f"🚀 Updated rows: {result.rowcount}")


🔍 Comparing Atlin School — DewPt
✅ No updates needed.

🔍 Comparing Atlin School — Gust Speed
✅ No updates needed.

🔍 Comparing Atlin School — RH
✅ No updates needed.

🔍 Comparing Atlin School — Rain
✅ No updates needed.

🔍 Comparing Atlin School — Sfc Temp
✅ No updates needed.

🔍 Comparing Atlin School — Soil Temp
✅ No updates needed.

🔍 Comparing Atlin School — Soil Temp 50 cm
✅ No updates needed.

🔍 Comparing Atlin School — Soil Temp 75 cm
✅ No updates needed.

🔍 Comparing Atlin School — Solar Radiation
✅ No updates needed.

🔍 Comparing Atlin School — Wind Direction
✅ No updates needed.

🔍 Comparing Atlin School — Wind Speed
✅ No updates needed.

🔍 Comparing BarrenWx — DewPt
✅ No updates needed.

🔍 Comparing BarrenWx — DewPt 30 cm
✅ No updates needed.

🔍 Comparing BarrenWx — Gust Speed
✅ No updates needed.

🔍 Comparing BarrenWx — Pressure
✅ No updates needed.

🔍 Comparing BarrenWx — RH
✅ No updates needed.

🔍 Comparing BarrenWx — RH 30 cm
✅ No updates needed.

🔍 Comparing BarrenWx —